# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata.get('name')}: {metadata.get('description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we list the available record sets, then fields within each record set, using their `@id` values.

In [ ]:
# List available record sets and their structure using @id references
record_sets = dataset.metadata.record_sets()
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}, Name: {rs.get('name', 'N/A')}")
    fields = rs.get('fields', [])
    for f in fields:
        print(f"  Field @id: {f['@id']} | Name: {f.get('name', 'N/A')} | DataType: {f.get('dataType', 'N/A')}")
    columns = rs.get('columns', [])
    for c in columns:
        print(f"  Column @id: {c['@id']} | Name: {c.get('name', 'N/A')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract all available record sets and load them into pandas DataFrames, referencing each by its `@id`.

In [ ]:
dataframes = {}
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets()]

for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet @id: {record_set_id} with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Could not load RecordSet @id: {record_set_id}, reason: {e}")

# Display a preview of one main record set
if len(dataframes) > 0:
    main_rs_id = list(dataframes.keys())[0]
    print(f"Preview of record set @id: {main_rs_id}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's demonstrate filtering for a numeric field and normalizing it, referencing all fields by their `@id`.

In [ ]:
# Find a numeric field (@id) for analysis by inspecting the main record set fields
main_rs_id = list(dataframes.keys())[0] if len(dataframes) > 0 else None
if main_rs_id:
    df = dataframes[main_rs_id]
    numeric_fields = []
    # Attempt to find typical numeric columns
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_fields.append(col)
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Use first numeric
        print(f"Using numeric field '@id': {numeric_field_id} for EDA.")
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try grouping by a categorical field (@id)
        group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by '{group_field_id}':")
            display(grouped_df.head())
    else:
        print("No numeric fields found in the record set.")
else:
    print("No record set found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot a histogram for the numeric field and a bar chart grouped by a categorical field, again referencing by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # Bar by group
    if group_fields:
        plt.figure(figsize=(7,4))
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset contains clinicopathological and molecular characteristics of cancer survivors with second primary colorectal cancer, with detailed fields referenced by their `@id`.
- Using `mlcroissant`, we dynamically loaded metadata and record sets for exploration and performed filtering, normalization, grouping, and visualization.
- All references respected Croissant schemas by using `@id` values throughout for reproducibility and schema-aware processing.